# 08. Synthetic pretrain V2

Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Пре-трейн голов XRD — v2 (модель ×8 больше)

Отличия от v1:
- **11.3M параметров** (v1 — 1.37M): каналы ×3, trunk 1536;
- batch 192, 0.24 с/шаг 15 эпох
- **повышенный вес лосса углов решётки** (w_ang=2.0): в v1 углы были самым слабым местом;
- сплит и стандартизации — те же честные (98/2 по формулам, seed 42), чтобы сравнение v1↔v2 было корректным.

**Режимы** (MODE): `test` — 100k подвыборка, 1 эпоха; `full` — все 460k, 15 эпох.

In [2]:
import json
import math
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0))

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [3]:
MODE = 'full'   # 'test' | 'full'

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
SYNTH_PARQUET = BASE / 'data' / 'clean' / 'df_synth_summary_final_clean.parquet'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

GRID_N = 4096
W = 3              # ширина модели (v1 было 1)
SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
W_LAT, W_VOL, W_SG, W_SYS, W_EL, W_ANG = 1.0, 0.5, 1.0, 1.0, 1.0, 2.0

HP = dict(
    batch=192,
    lr=3e-4,
    wd=1e-4,
    warmup=500,
    clip=1.0,
    epochs={'test': 1, 'full': 15}[MODE],
    train_subset={'test': 100_000, 'full': None}[MODE],
    val_subset={'test': 4000, 'full': None}[MODE],
    val_every={'test': None, 'full': 1500}[MODE],
)
print('MODE =', MODE, '| HP:', HP)

MODE = full | HP: {'batch': 192, 'lr': 0.0003, 'wd': 0.0001, 'warmup': 500, 'clip': 1.0, 'epochs': 15, 'train_subset': None, 'val_subset': None, 'val_every': 1500}


In [ ]:
# ---------- данные (те же артефакты и сплит, что в v1) ----------
index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
pre = index[index['split_role'] == 'pretrain'].reset_index(drop=True)

syn = pd.read_parquet(SYNTH_PARQUET)[
    ['sample_id', 'lattice_a', 'lattice_b', 'lattice_c',
     'alpha', 'beta', 'gamma', 'spacegroup_number',
     'crystal_system', 'elements_list', 'phase_compositions']
]
df = pre.merge(syn, on='sample_id', how='left')
assert df['lattice_a'].notna().all()

def to_list(v):
    if isinstance(v, str):
        try:
            return json.loads(v)
        
        except Exception:
            return []
        
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    return []

df['elements'] = df['elements_list'].apply(to_list)
df['formula'] = df['phase_compositions'].apply(
    lambda p: (p[0] if isinstance(p, (list, tuple, np.ndarray)) and len(p) > 0
               else (p if isinstance(p, str) else 'NA'))
)

cnt = Counter()

for els in df['elements']:
    cnt.update(els)

VOCAB = sorted(cnt)
EL_IDX = {e: i for i, e in enumerate(VOCAB)}

abc = df[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang = df[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
V = abc.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
df['lat6'] = list(np.hstack([np.log(abc), ang]))
df['logV'] = np.log(V)

# сплит и стандартизации берём СОХРАНЁННЫЕ из v1 - сравнение будет корректным

splits_v1 = pd.read_parquet(OUT / 'splits_pretrain.parquet')
df = df.merge(splits_v1, on='sample_id', how='left')
assert df['split'].notna().all()
stats_v1 = json.loads((OUT / 'pretrain_stats.json').read_text())
LAT_MEAN = np.array(stats_v1['lat_mean'])
LAT_STD = np.array(stats_v1['lat_std'])
VOL_MEAN, VOL_STD = stats_v1['vol_mean'], stats_v1['vol_std']
assert stats_v1['vocab'] == VOCAB, 'словарь элементов изменился'
print('использую сплиты и стандартизации v1 (сравнимость)')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

print(f'train {len(train_df)} / val {len(val_df)}')

использую сплиты и стандартизации v1 (сравнимость)
train 451027 / val 9106


In [ ]:
# ---------- датасет ----------
N_TOTAL = len(index)

X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL, GRID_N))

class PretrainDS(Dataset):
    def __init__(self, frame):
        self.row = frame['row_idx'].to_numpy(np.int64)
        lam1 = frame['lambda_1'].to_numpy(np.float32)
        lam2 = frame['lambda_2'].to_numpy(np.float32)
        self.lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                             np.isfinite(lam2).astype(np.float32)], 1)
        self.lat6 = (np.stack(frame['lat6'].to_numpy()) - LAT_MEAN) / LAT_STD
        self.latm = np.ones(len(frame), np.float32)
        self.vol = (frame['logV'].to_numpy(np.float32) - VOL_MEAN) / VOL_STD
        sg = frame['spacegroup_number'].to_numpy(float)
        self.sgm = np.isfinite(sg).astype(np.float32)
        self.sg = np.nan_to_num(sg).astype(np.int64) - 1
        sysmap = {s: i for i, s in enumerate(SYSTEMS)}
        sys_raw = frame['crystal_system'].map(sysmap)
        self.sysm = sys_raw.notna().to_numpy(np.float32)
        self.sys = sys_raw.fillna(0).to_numpy(np.int64)
        n = len(frame)
        self.el = np.zeros((n, len(VOCAB)), np.float32)

        for i, els in enumerate(frame['elements']):
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    self.el[i, j] = 1.0

        self.elm = np.ones(n, np.float32)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        r = self.row[i]
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[r]
        x[1] = M_MM[r]
        return (torch.from_numpy(x), torch.from_numpy(self.lam[i]),
                torch.from_numpy(self.lat6[i]), torch.tensor(self.latm[i]),
                torch.tensor(self.sg[i]), torch.tensor(self.sgm[i]),
                torch.tensor(self.sys[i]), torch.tensor(self.sysm[i]),
                torch.from_numpy(self.el[i]), torch.tensor(self.elm[i]),
                torch.tensor(self.vol[i]), torch.tensor(self.latm[i]))

def make_loaders():
    tr, va = train_df, val_df

    if HP['train_subset'] and HP['train_subset'] < len(tr):
        tr = tr.sample(HP['train_subset'], random_state=SEED)

    if HP['val_subset'] and HP['val_subset'] < len(va):
        va = va.sample(HP['val_subset'], random_state=SEED)

    ld_tr = DataLoader(PretrainDS(tr), batch_size=HP['batch'], shuffle=True,
                       drop_last=True, pin_memory=True)
    ld_va = DataLoader(PretrainDS(va), batch_size=HP['batch'], shuffle=False, pin_memory=True)
    
    return ld_tr, ld_va

train_loader, val_loader = make_loaders()
print('train batches:', len(train_loader), '| val batches:', len(val_loader))

train batches: 2349 | val batches: 48


In [ ]:
# ---------- модель v2: W=3 ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)

        if cin == cout and stride == 1:
            self.skip = nn.Identity()

        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))

        return F.gelu(h + self.skip(x))

class XRDNetV2(nn.Module):
    def __init__(self, n_el, w=3):
        super().__init__()
        c = [32 * w, 48 * w, 64 * w, 96 * w, 128 * w, 192 * w, 256 * w]
        self.stem = nn.Sequential(nn.Conv1d(2, c[0], 15, padding=7, bias=False),
                                  nn.GroupNorm(8, c[0]), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(c[i], c[i + 1], stride=2)
                                      for i in range(6)])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16 * w), nn.GELU(), nn.Linear(16 * w, 16 * w))
        self.trunk = nn.Sequential(nn.Linear(c[-1] + 16 * w, 512 * w), nn.GELU(),
                                   nn.Linear(512 * w, 512 * w), nn.GELU())
        self.head_lat = nn.Linear(512 * w, 6)
        self.head_vol = nn.Linear(512 * w, 1)
        self.head_sg = nn.Linear(512 * w, 230)
        self.head_sys = nn.Linear(512 * w, 7)
        self.head_el = nn.Linear(512 * w, n_el)

    def forward(self, x, lam):
        f = self.stem(x)
        f = self.blocks(f)
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)

        return dict(lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z))

model = XRDNetV2(len(VOCAB), w=W).to(DEVICE)

print('параметров: %.2f M' % (sum(p.numel() for p in model.parameters()) / 1e6))

параметров: 11.27 M


In [ ]:
# ---------- лоссы (углы решётки - повышенный вес) ----------

def masked_l1(pred, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(())
    
    return F.smooth_l1_loss(pred[m], tgt[m])

def masked_l1_split(pred, tgt, mask):

    """Отдельно длины (0:3) и углы (3:6) - углам повышенный вес."""

    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(()), pred.new_zeros(())
    
    l_len = F.smooth_l1_loss(pred[m][:, :3], tgt[m][:, :3])
    l_ang = F.smooth_l1_loss(pred[m][:, 3:], tgt[m][:, 3:])

    return l_len, l_ang

def masked_ce(logits, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return logits.new_zeros(())
    
    return F.cross_entropy(logits[m], tgt[m].long())

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    l_len, l_ang = masked_l1_split(out['lat'], lat, latm)
    losses = dict(
        lat=l_len,
        ang=l_ang,
        vol=masked_l1(out['vol'], vol, volm),
        sg=masked_ce(out['sg'], sg, sgm),
        sys=masked_ce(out['sys'], sys_, sysm),
        el=F.binary_cross_entropy_with_logits(out['el'], el),
    )
    weighted = (W_LAT * losses['lat'] + W_ANG * losses['ang'] + W_VOL * losses['vol']
                + W_SG * losses['sg'] + W_SYS * losses['sys'] + W_EL * losses['el'])
    
    return losses, weighted

In [ ]:
# ---------- метрики и цикл обучения ----------

@torch.no_grad()

def evaluate(model, loader):
    model.eval()
    sums = defaultdict(float)
    n = 0
    sg_ok = sg_n = sys_ok = sys_n = 0
    tp = fp = fn = 0
    lat_abs = np.zeros(6)
    lat_n = 0
    vol_rel = []

    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]
        x, lam = batch[0], batch[1]

        with torch.autocast('cuda', dtype=torch.float16):
            out = model(x, lam)

        losses, _ = compute_losses(out, batch)
        bs = len(x)
        n += bs

        for k, v in losses.items():
            sums[k] += float(v) * bs
        sums['weighted'] += float(sum(losses.values())) * bs

        sg_m = batch[5] > 0

        if sg_m.any():
            sg_ok += (out['sg'][sg_m].argmax(1) == batch[4][sg_m].long()).sum().item()
            sg_n += int(sg_m.sum())
        sys_m = batch[7] > 0

        if sys_m.any():
            sys_ok += (out['sys'][sys_m].argmax(1) == batch[6][sys_m].long()).sum().item()
            sys_n += int(sys_m.sum())

        pred_bin = (out['el'] > 0).float()
        tp += float((pred_bin * batch[8]).sum().item())
        fp += float((pred_bin * (1 - batch[8])).sum().item())
        fn += float(((1 - pred_bin) * batch[8]).sum().item())
        latm = batch[3] > 0

        if latm.any():
            p = out['lat'][latm].float().cpu().numpy() * LAT_STD + LAT_MEAN
            t = batch[2][latm].cpu().numpy() * LAT_STD + LAT_MEAN
            p[:, :3] = np.exp(p[:, :3]); t[:, :3] = np.exp(t[:, :3])
            lat_abs += np.abs(p - t).sum(0)
            lat_n += len(p)
            pv = out['vol'][latm].float().cpu().numpy() * VOL_STD + VOL_MEAN
            tv = batch[10][latm].cpu().numpy() * VOL_STD + VOL_MEAN
            vol_rel.append(np.abs(np.exp(pv) - np.exp(tv)) / np.exp(tv))

    model.train()
    prec = tp / max(tp + fp, 1e-9)
    rec = tp / max(tp + fn, 1e-9)
    res = {k: v / n for k, v in sums.items()}
    res['sg_acc'] = sg_ok / max(sg_n, 1)
    res['sys_acc'] = sys_ok / max(sys_n, 1)
    res['el_f1_micro'] = 2 * prec * rec / max(prec + rec, 1e-9)
    if lat_n:
        mae = lat_abs / lat_n
        res.update(mae_a_A=mae[0], mae_b_A=mae[1], mae_c_A=mae[2],
                   mae_ang_deg=mae[3:].mean(),
                   vol_rel_err=float(np.concatenate(vol_rel).mean()))
    res['n'] = n

    return res

def fmt_metrics(m):
    keys = ['weighted', 'lat', 'ang', 'sg', 'sys', 'el',
            'sg_acc', 'sys_acc', 'el_f1_micro',
            'mae_a_A', 'mae_b_A', 'mae_c_A', 'mae_ang_deg', 'vol_rel_err']
    
    return {k: round(float(m[k]), 4) for k in keys if k in m}

opt = torch.optim.AdamW(model.parameters(), lr=HP['lr'], weight_decay=HP['wd'])
scaler = torch.amp.GradScaler('cuda')

def make_sched(total_steps):
    def fn(step):
        if step < HP['warmup']:
            return step / max(HP['warmup'], 1)
        p = (step - HP['warmup']) / max(total_steps - HP['warmup'], 1)
        return 0.5 * (1.0 + math.cos(math.pi * min(p, 1.0)))
    
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def run_training(tag):
    total_steps = HP['epochs'] * len(train_loader)
    sched = make_sched(total_steps)
    best = float('inf')
    t0 = time.time()
    step = 0
    runma = defaultdict(float)
    log_rows = []

    for epoch in range(HP['epochs']):
        for batch in train_loader:
            batch = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(batch[0], batch[1])
                losses, total = compute_losses(out, batch)

            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            for k, v in losses.items():
                runma[k] += float(v)

            runma['weighted'] += float(total)

            if step % 100 == 0 or step == 1:
                avg = {k: v / (100 if step > 1 else 1) for k, v in runma.items()}
                runma = defaultdict(float)
                msg = ' | '.join(f'{k} {avg[k]:.4f}'
                                 for k in ['weighted', 'lat', 'ang', 'sg', 'sys', 'el'])
                print(f'epoch {epoch+1} step {step:5d}/{total_steps} | {msg} | '
                      f'{(time.time()-t0)/step:.2f} с/шаг', flush=True)
                log_rows.append(dict(step=step, epoch=epoch + 1, **avg))

            if HP['val_every'] and step % HP['val_every'] == 0:
                m = evaluate(model, val_loader)
                print('  VAL:', fmt_metrics(m), flush=True)

                if m['weighted'] < best:
                    best = m['weighted']
                    torch.save(model.state_dict(), CKPT_DIR / f'{tag}_best.pt')

    torch.save(model.state_dict(), CKPT_DIR / f'{tag}_last.pt')
    print(f'готово за {(time.time()-t0)/60:.1f} мин | шагов {step}')
    
    return log_rows

In [ ]:
# ---------- запуск ----------
base_metrics = evaluate(model, val_loader)

print('БАЗЛАЙН (до обучения):')

print(fmt_metrics(base_metrics))
print()
train_log = run_training(f'pretrain_v2_{MODE}')

final_metrics = evaluate(model, val_loader)
print()

print('ИТОГ (после обучения):')
print(fmt_metrics(final_metrics))

БАЗЛАЙН (до обучения):
{'weighted': 9.3651, 'lat': 0.4508, 'ang': 0.3493, 'sg': 5.4469, 'sys': 1.9696, 'el': 0.6923, 'sg_acc': 0.0001, 'sys_acc': 0.1031, 'el_f1_micro': 0.1087, 'mae_a_A': 3.4545, 'mae_b_A': 3.8171, 'mae_c_A': 5.2921, 'mae_ang_deg': 8.1751, 'vol_rel_err': 1.9907}

epoch 1 step     1/35235 | weighted 9.4939 | lat 0.4506 | ang 0.3584 | sg 5.4441 | sys 1.9740 | el 0.6924 | 3.01 с/шаг
epoch 1 step   100/35235 | weighted 7.1782 | lat 0.4156 | ang 0.3439 | sg 3.8815 | sys 1.5650 | el 0.4226 | 0.35 с/шаг
epoch 1 step   200/35235 | weighted 6.0817 | lat 0.4254 | ang 0.3534 | sg 3.1185 | sys 1.5036 | el 0.1147 | 0.33 с/шаг
epoch 1 step   300/35235 | weighted 6.1273 | lat 0.4271 | ang 0.3529 | sg 3.1585 | sys 1.5095 | el 0.1126 | 0.33 с/шаг
epoch 1 step   400/35235 | weighted 5.6263 | lat 0.3556 | ang 0.3467 | sg 2.9380 | sys 1.3853 | el 0.1074 | 0.33 с/шаг
epoch 1 step   500/35235 | weighted 5.2738 | lat 0.3073 | ang 0.3458 | sg 2.7697 | sys 1.3041 | el 0.0989 | 0.33 с/шаг
epoch

In [ ]:
# ---------- сравнение v1 vs v2 на одном и том же вале ----------

v1_final = {'weighted': 3.9697, 'lat': 0.291, 'sg_acc': 0.3745, 'sys_acc': 0.5424,
            'el_f1_micro': 0.6311, 'mae_a_A': 2.6616, 'mae_ang_deg': 7.6181,
            'vol_rel_err': 0.5093}  # из прогона v1 (MODE=full)
rows = []

for k in ['sg_acc', 'sys_acc', 'el_f1_micro', 'mae_a_A', 'mae_ang_deg', 'vol_rel_err']:
    rows.append(dict(metric=k, v1=v1_final.get(k), v2=final_metrics.get(k)))

print(pd.DataFrame(rows).round(4).to_string(index=False))

lg = pd.DataFrame(train_log)
fig, ax = plt.subplots(figsize=(9, 4.5))

for k in ['weighted', 'lat', 'ang', 'sg', 'sys', 'el']:
    if k in lg:
        ax.plot(lg['step'], lg[k], label=k, lw=1.6)

ax.set_xlabel('шаг'); ax.set_ylabel('loss'); ax.set_yscale('log')
ax.grid(alpha=0.3); ax.legend(ncol=3, fontsize=9)
ax.set_title(f'Пре-трейн v2 ({sum(p.numel() for p in model.parameters())/1e6:.1f}M), MODE={MODE}')

plt.tight_layout()
plt.savefig(OUT / f'pretrain_v2_{MODE}_losses.png', dpi=130)

print('график:', OUT / f'pretrain_v2_{MODE}_losses.png')

     metric     v1     v2
     sg_acc 0.3745 0.4026
    sys_acc 0.5424 0.5695
el_f1_micro 0.6311 0.6504
    mae_a_A 2.6616 2.6922
mae_ang_deg 7.6181 7.2933
vol_rel_err 0.5093 0.4793
график: D:\Users\user\Desktop\DS_XRD_project\outputs\pretrain_v2_full_losses.png
